In [1]:
import os

print("appointments.csv exists:", os.path.exists("/content/followup_notes.csv"))

appointments.csv exists: True


In [2]:

# Part 4 — Setup and inspect follow-up notes

!pip install -q google-genai pydantic

import pandas as pd
import json

notes_df = pd.read_csv("followup_notes.csv")

print("Dataset shape:", notes_df.shape)
print("Columns:", notes_df.columns.tolist())
print("\nFirst 10 notes:")
print(notes_df.head(10).to_string(index=False))

print("\nBlank notes in first 10:",
      notes_df.head(10)["note_text"].isna().sum())


Dataset shape: (30, 3)
Columns: ['note_id', 'appointment_id', 'note_text']

First 10 notes:
note_id appointment_id                                                                                                                     note_text
   N001          A0872         Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
   N002          A0897   Lives far from the clinic and could not find any transport that morning. Says the clinic is too far for a routine check-up.
   N003          A0526 Patient felt the consultation fee was too high this month and decided to skip. Asked if there is any discount for follow-ups.
   N004          A0926                                                                                                                           NaN
   N005          A0534                                            Had high fever and body ache on the day, stayed in bed. Will visit once recovered.
   N006       

In [18]:

# Task 4.1 — Gemini API connection setup

from google import genai
from google.genai import types
from getpass import getpass

# Enter your key privately at runtime
API_KEY = getpass("Enter Gemini API key: ")

client = genai.Client(api_key=API_KEY)

# List available Flash models
print("Available Gemini Flash models:")

for model in client.models.list():
    if "flash" in model.name.lower():
        print(model.name)


Enter Gemini API key: ··········
Available Gemini Flash models:
models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [7]:

# ==================================================
# TASK 4.1 — ONE VALIDATED GEMINI API CALL
# ==================================================

import json
from google import genai
from google.genai import types

# Available Gemini Flash model
MODEL_NAME = "gemini-3.8-flash"

# Allowed root-cause categories
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# Required JSON response schema
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

# System instruction
SYSTEM_TEXT = (
    "You are a healthcare operations analyst. "
    "Analyse patient follow-up notes accurately. "
    "Do not invent information. "
    "Return only valid JSON."
)


# Function to analyse one patient note
def analyse_note(note_text):

    # CRAFT prompt
    prompt = f"""
Context:
A clinic wants to understand why patients
miss their scheduled appointments.

Role:
You are a healthcare operations analyst.

Action:
Analyse the patient note and identify
the most likely reason for missing the appointment.

Choose root_cause ONLY from:
{", ".join(ALLOWED_CAUSES)}

If the reason is unclear, use "other".
Confidence must be between 0 and 1.

Tone:
Professional, neutral and concise.

Format:
Return JSON with exactly four fields:
1. root_cause
2. confidence
3. summary
4. recommended_action

Summary must contain no more than 25 words.

Patient note:
{note_text}
"""

    # Gemini API request
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    # Parse JSON response
    data = json.loads(response.text)

    return data


# ==================================================
# TEST ON NOTE N001
# ==================================================

note_n001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

result_n001 = analyse_note(note_n001)

print("Task 4.1 — N001 Analysis")
print(json.dumps(result_n001, indent=2))


Task 4.1 — N001 Analysis
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.98,
  "summary": "Patient forgot the visit after failing to receive a reminder message and requested to rebook next week.",
  "recommended_action": "Reschedule the patient for next week and verify contact details to ensure automated appointment reminders are delivered."
}


In [8]:

# Task 4.2(a) — Pydantic validation

from pydantic import BaseModel, Field, field_validator

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

    @field_validator("root_cause")
    @classmethod
    def check_root_cause(cls, value):
        if value not in ALLOWED_CAUSES:
            raise ValueError(
                f"Invalid root cause: {value}"
            )
        return value


# Validate the existing N001 result
validated_n001 = FollowupInsight(**result_n001)

print("N001 validation successful!")
print(validated_n001.model_dump())


N001 validation successful!
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient forgot the visit after failing to receive a reminder message and requested to rebook next week.', 'recommended_action': 'Reschedule the patient for next week and verify contact details to ensure automated appointment reminders are delivered.'}


In [9]:

# Task 4.2(b–d) — Process and save 10 notes

import time
import pandas as pd
from google.colab import files

records = []
api_calls_42 = 0

for _, row in notes_df.head(10).iterrows():

    note_id = row["note_id"]
    note_text = row["note_text"]

    record = {
        "note_id": note_id,
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Skip blank notes without an API call
    if pd.isna(note_text) or str(note_text).strip() == "":
        record.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank"
        })

    else:
        # Wait between API requests
        if api_calls_42 > 0:
            time.sleep(13)

        try:
            # Count each API request attempted
            api_calls_42 += 1
            data = analyse_note(str(note_text))

            # Preserve the returned dictionary as JSON
            record["raw_response"] = json.dumps(
                data, ensure_ascii=False
            )

            # Copy fields without repairing invalid replies
            for field in [
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            ]:
                record[field] = data.get(field, "")

            # Validate response
            FollowupInsight(**data)
            record["validation_status"] = "valid"

        except Exception as error:
            record["validation_status"] = "invalid"
            print(f"{note_id} error: {error}")

    records.append(record)
    print(f"{note_id}: {record['validation_status']}")

# Save exact required column order
required_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

analysed_df = pd.DataFrame(
    records, columns=required_columns
)

analysed_df.to_csv(
    "notes_analysed.csv",
    index=False
)

print("\nFinal shape:", analysed_df.shape)
print("API requests attempted:", api_calls_42)

print("\nValidation status counts:")
print(analysed_df["validation_status"].value_counts())

print("\nRoot cause results:")
print(
    analysed_df[
        ["note_id", "root_cause", "validation_status"]
    ].to_string(index=False)
)

files.download("notes_analysed.csv")


N001: valid
N002: valid
N003: valid
N004: skipped_blank
N005 error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N005: invalid
N006 error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N006: invalid
N007: valid
N008 error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N008: invalid
N009 error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N009: invalid
N010 error: 503 UNAVAILABLE. {'error': {'code': 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:

# Task 4.2 — Retry N005 only

MODEL_NAME = "gemini-3.1-flash-lite"

note_n005 = notes_df.loc[
    notes_df["note_id"] == "N005",
    "note_text"
].iloc[0]

try:
    result_n005 = analyse_note(note_n005)
    validated_n005 = FollowupInsight(**result_n005)

    print("N005: valid")
    print(json.dumps(result_n005, indent=2))

except Exception as error:
    print("N005 retry failed:")
    print(error)


N005: valid
{
  "root_cause": "illness_or_emergency",
  "confidence": 1.0,
  "summary": "Patient missed the appointment due to high fever and body aches, requiring bed rest.",
  "recommended_action": "Reschedule the appointment once the patient has fully recovered."
}


In [11]:

# Task 4.2 — Retry remaining failed notes

import time
import json

MODEL_NAME = "gemini-3.1-flash-lite"

# Save the successful N005 retry
idx = analysed_df.index[
    analysed_df["note_id"] == "N005"
][0]

for field in [
    "root_cause", "confidence",
    "summary", "recommended_action"
]:
    analysed_df.at[idx, field] = result_n005[field]

analysed_df.at[idx, "validation_status"] = "valid"
analysed_df.at[idx, "raw_response"] = json.dumps(result_n005)

# Retry only the remaining failures
retry_ids = ["N006", "N008", "N009", "N010"]

api_calls_retry = 1  # N005 retry already completed

for note_id in retry_ids:

    note_text = notes_df.loc[
        notes_df["note_id"] == note_id,
        "note_text"
    ].iloc[0]

    time.sleep(13)

    api_calls_retry += 1

    try:
        data = analyse_note(note_text)

        idx = analysed_df.index[
            analysed_df["note_id"] == note_id
        ][0]

        analysed_df.at[idx, "raw_response"] = json.dumps(data)

        for field in [
            "root_cause", "confidence",
            "summary", "recommended_action"
        ]:
            analysed_df.at[idx, field] = data.get(field, "")

        FollowupInsight(**data)

        analysed_df.at[idx, "validation_status"] = "valid"

        print(f"{note_id}: valid — {data['root_cause']}")

    except Exception as error:
        print(f"{note_id}: retry failed — {error}")

# Save updated CSV
analysed_df.to_csv("notes_analysed.csv", index=False)

print("\nFinal shape:", analysed_df.shape)

print("\nValidation status counts:")
print(analysed_df["validation_status"].value_counts())

print("\nRoot cause results:")
print(
    analysed_df[
        ["note_id", "root_cause", "validation_status"]
    ].to_string(index=False)
)

print("\nAdditional API requests:", api_calls_retry)


N006: valid — work_or_schedule
N008: valid — distance_or_transport
N009: valid — other
N010: valid — work_or_schedule

Final shape: (10, 8)

Validation status counts:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Root cause results:
note_id            root_cause validation_status
   N001 forgot_or_no_reminder             valid
   N002 distance_or_transport             valid
   N003           cost_or_fee             valid
   N004                 other     skipped_blank
   N005  illness_or_emergency             valid
   N006      work_or_schedule             valid
   N007 forgot_or_no_reminder             valid
   N008 distance_or_transport             valid
   N009                 other             valid
   N010      work_or_schedule             valid

Additional API requests: 5


In [12]:

# Task 4.3(a) — Deliberately invalid responses

from pydantic import ValidationError

# Test 1: Confidence above the allowed maximum
bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient missed appointment due to no reminder.",
    "recommended_action": "Verify reminder delivery."
}

# Test 2: Missing required field
bad_reply_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.9,
    "summary": "Patient could not find transportation."
}

for test_name, bad_reply in [
    ("Test 1: Confidence = 1.7", bad_reply_1),
    ("Test 2: Missing recommended_action", bad_reply_2)
]:
    try:
        FollowupInsight(**bad_reply)
        print(test_name, "— Unexpectedly passed")

    except ValidationError as error:
        print(f"\n{test_name} — Correctly rejected")
        print(error)



Test 1: Confidence = 1.7 — Correctly rejected
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Test 2: Missing recommended_action — Correctly rejected
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'distance_...t find transportation.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:

# Task 4.3(b) — Basic prompt V1

import time
import json

MODEL_NAME = "gemini-3.1-flash-lite"

v1_results = []

for i, note_id in enumerate(["N007", "N009"]):

    if i > 0:
        time.sleep(13)

    note_text = notes_df.loc[
        notes_df["note_id"] == note_id,
        "note_text"
    ].iloc[0]

    # V1: basic prompt
    prompt_v1 = f"Classify this note:\n{note_text}"

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_v1,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    data = json.loads(response.text)

    v1_results.append({
        "note_id": note_id,
        "prompt_version": "V1",
        "root_cause": data.get("root_cause", ""),
        "allowed_category": data.get("root_cause") in ALLOWED_CAUSES
    })

    print(
        note_id,
        "V1:",
        data.get("root_cause"),
        "| Allowed:",
        data.get("root_cause") in ALLOWED_CAUSES
    )


In [14]:

# Task 4.3(b) — Retry V1 for N007 only

import json
from google.genai import types

MODEL_NAME = "gemini-3.1-flash-lite"

note_n007 = notes_df.loc[
    notes_df["note_id"] == "N007",
    "note_text"
].iloc[0]

try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=f"Classify this note:\n{note_n007}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    result_v1_n007 = json.loads(response.text)

    print("N007 — V1 result:")
    print(json.dumps(result_v1_n007, indent=2))

    print(
        "Allowed root cause:",
        result_v1_n007.get("root_cause") in ALLOWED_CAUSES
    )

except Exception as error:
    print("API request failed:")
    print(error)


N007 — V1 result:
{
  "root_cause": "Patient forgot the appointment due to lack of SMS reminder.",
  "confidence": 0.95,
  "summary": "The patient missed their appointment because they did not receive an SMS notification and is now requesting to reschedule.",
  "recommended_action": "Verify patient contact details in the system, ensure SMS notification settings are enabled, and assist the patient with rebooking the appointment."
}
Allowed root cause: False


In [15]:

# Task 4.3(b) — V1 for N009

note_n009 = notes_df.loc[
    notes_df["note_id"] == "N009",
    "note_text"
].iloc[0]

try:
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=f"Classify this note:\n{note_n009}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    result_v1_n009 = json.loads(response.text)

    print("N009 — V1 result:")
    print(json.dumps(result_v1_n009, indent=2))

    print(
        "Allowed root cause:",
        result_v1_n009.get("root_cause") in ALLOWED_CAUSES
    )

except Exception as error:
    print("API request failed:")
    print(error)


N009 — V1 result:
{
  "root_cause": "Patient unreachable due to phone being switched off",
  "confidence": 1.0,
  "summary": "Three unsuccessful attempts were made to contact the patient via phone, but the device was powered off.",
  "recommended_action": "Schedule a follow-up call for a later time or send a secure message/letter to the patient."
}
Allowed root cause: False


In [16]:

# Task 4.3(b–c) — Prompt comparison and API usage

import pandas as pd

comparison_df = pd.DataFrame([
    {
        "note_id": "N007",
        "prompt": "V1 - Basic",
        "root_cause": result_v1_n007["root_cause"]
    },
    {
        "note_id": "N007",
        "prompt": "V2 - CRAFT",
        "root_cause": analysed_df.loc[
            analysed_df["note_id"] == "N007", "root_cause"
        ].iloc[0]
    },
    {
        "note_id": "N009",
        "prompt": "V1 - Basic",
        "root_cause": result_v1_n009["root_cause"]
    },
    {
        "note_id": "N009",
        "prompt": "V2 - CRAFT",
        "root_cause": analysed_df.loc[
            analysed_df["note_id"] == "N009", "root_cause"
        ].iloc[0]
    }
])

comparison_df["allowed_category"] = (
    comparison_df["root_cause"].isin(ALLOWED_CAUSES)
)

print("Task 4.3(b) — Prompt comparison")
print(comparison_df.to_string(index=False))

print("\nRecommended prompt: V2 - CRAFT")

# Count explicit API requests, including failed requests
task_41_calls = 1
task_42_calls = 9 + 5
task_43_calls = 3  # 1 failed V1 request + 2 successful V1 requests

total_calls = task_41_calls + task_42_calls + task_43_calls

print("\nTask 4.3(c) — API request accounting")
print("Task 4.1:", task_41_calls)
print("Task 4.2:", task_42_calls)
print("Task 4.3:", task_43_calls)
print("Total explicit requests:", total_calls)


Task 4.3(b) — Prompt comparison
note_id     prompt                                                  root_cause  allowed_category
   N007 V1 - Basic Patient forgot the appointment due to lack of SMS reminder.             False
   N007 V2 - CRAFT                                       forgot_or_no_reminder              True
   N009 V1 - Basic         Patient unreachable due to phone being switched off             False
   N009 V2 - CRAFT                                                       other              True

Recommended prompt: V2 - CRAFT

Task 4.3(c) — API request accounting
Task 4.1: 1
Task 4.2: 14
Task 4.3: 3
Total explicit requests: 18


In [17]:

# Task 4.4(a) — Merge notes with appointments

import pandas as pd
from google.colab import files

uploaded = files.upload()  # Select appointments_clean.csv

appointments_df = pd.read_csv("appointments_clean.csv")

merged_df = analysed_df.merge(
    appointments_df[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left",
    validate="one_to_one"
)

print("Merged table shape:", merged_df.shape)

print("\nRoot cause vs reminder sent:")
cross_tab = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)
print(cross_tab)

print("\nReminder counts:")
print(merged_df["reminder_sent"].value_counts(dropna=False))

print("\nForgot/no-reminder cases:")
print(
    merged_df.loc[
        merged_df["root_cause"] == "forgot_or_no_reminder",
        ["note_id", "appointment_id", "reminder_sent"]
    ].to_string(index=False)
)


Saving appointments_clean.csv to appointments_clean.csv
Merged table shape: (10, 9)

Root cause vs reminder sent:
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

Reminder counts:
reminder_sent
No     5
Yes    5
Name: count, dtype: int64

Forgot/no-reminder cases:
note_id appointment_id reminder_sent
   N001          A0872            No
   N007          A0333            No
